In [6]:
import json
from typing import Dict, Any, List

In [7]:
# --- Step 1: Define the Real Local Python Hardware Tools ---

def adjust_omen_fan_speed(speed_rpm: int) -> Dict[str, Any]:
    """Adjusts cooling performance parameters on the development laptop."""
    print(f"\n[HARDWARE EXECUTION] Sending PWM signals to fans. Setting RPM to: {speed_rpm}")
    return {"status": "SUCCESS", "current_fan_speed_rpm": speed_rpm, "thermal_profile": "PERFORMANCE"}

In [8]:
# --- Step 2: Define the JSON Schema Blueprints for the LLM ---
# This is exactly what is delivered to providers like OpenAI/Anthropic 
AVAILABLE_TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "adjust_omen_fan_speed",
            "description": "Modifies the active internal fan rotation speed on the hardware cooling array.",
            "parameters": {
                "type": "object",
                "properties": {
                    "speed_rpm": {
                        "type": "integer",
                        "description": "The target revolutions per minute. Safe limits are between 2000 and 6000."
                    }
                },
                "required": ["speed_rpm"]
            }
        }
    }
]

In [9]:
# --- Step 3: The Execution Tool Router Layer ---

class FunctionCallingRouter:
    """Simulates the application-side engine handling the LLM function calling handshake."""
    
    def simulate_llm_decision(self, user_intent: str) -> Dict[str, Any]:
        """Simulates an LLM parsing intent and selecting a native schema target payload."""
        print(f"[LLM Core] Analyzing query: '{user_intent}' against registered tool schemas...")
        
        # Simulating the exact structural JSON payload a model spits out when it triggers a tool
        if "cool" in user_intent.lower() or "fan" in user_intent.lower():
            mock_payload = {
                "role": "assistant",
                "tool_calls": [
                    {
                        "id": "call_abc123",
                        "type": "function",
                        "function": {
                            "name": "adjust_omen_fan_speed",
                            "arguments": "{\"speed_rpm\": 5500}" # Serialized JSON string outputted by model
                        }
                    }
                ]
            }
            return mock_payload
        return {"role": "assistant", "content": "I am not sure which hardware function to trigger."}

    def execute_pipeline(self, user_query: str):
        # 1. Ship prompt and tools to the model
        llm_response = self.simulate_llm_decision(user_query)
        
        # 2. Check if the model has made an explicit tool call request
        if "tool_calls" in llm_response:
            for tool_call in llm_response["tool_calls"]:
                function_info = tool_call["function"]
                function_name = function_info["name"]
                
                # PITFALL WARNING: The arguments come back from the model as a raw string!
                # You MUST deserialize them using json.loads() before using them in code.
                raw_arguments_str = function_info["arguments"]
                parsed_args = json.loads(raw_arguments_str)
                
                print(f"[ROUTER] Model requested execution of function: '{function_name}'")
                print(f"[ROUTER] Extracted arguments payload: {parsed_args}")
                
                # 3. Dynamic Execution Mapping Routing
                if function_name == "adjust_omen_fan_speed":
                    # Unpack the verified dict values directly into the actual function call
                    execution_result = adjust_omen_fan_speed(**parsed_args)
                    
                    # 4. Construct the final tool execution message to pass back into the loop
                    tool_message_payload = {
                        "role": "tool",
                        "tool_call_id": tool_call["id"],
                        "name": function_name,
                        "content": json.dumps(execution_result)
                    }
                    print(f"[ROUTER] Shifting transaction payload back to LLM chain:\n{json.dumps(tool_message_payload, indent=2)}")

In [10]:
# Execute our router loop
engine = FunctionCallingRouter()
engine.execute_pipeline("My GPU is throttling during compilation, spin up the fans to 5500 RPM immediately.")

[LLM Core] Analyzing query: 'My GPU is throttling during compilation, spin up the fans to 5500 RPM immediately.' against registered tool schemas...
[ROUTER] Model requested execution of function: 'adjust_omen_fan_speed'
[ROUTER] Extracted arguments payload: {'speed_rpm': 5500}

[HARDWARE EXECUTION] Sending PWM signals to fans. Setting RPM to: 5500
[ROUTER] Shifting transaction payload back to LLM chain:
{
  "role": "tool",
  "tool_call_id": "call_abc123",
  "name": "adjust_omen_fan_speed",
  "content": "{\"status\": \"SUCCESS\", \"current_fan_speed_rpm\": 5500, \"thermal_profile\": \"PERFORMANCE\"}"
}
